### ⚠️ Patched for Local Execution
This notebook was originally designed for Google Colab. It has been automatically patched:
- Google Colab-specific imports and `drive.mount()` calls have been commented out
- Colab file paths (`/content/drive/...`) have been replaced with relative paths (`./`)
- `!pip install` commands have been commented out (install packages in your venv instead)

**To run locally:** activate your Python virtual environment first, then run this notebook in VS Code or Jupyter.

In [ ]:
# [PATCHED] from google.colab import drive
# [PATCHED] drive.mount('./')

In [ ]:
%%bash
cp ./ ./

In [ ]:
%%bash
cd /content
unzip data.zip

In [ ]:
%%bash
cd ./
rm *.keras

In [ ]:
# [PATCHED] !pip install keras-cv tensorflow --upgrade

In [ ]:
# [PATCHED] !pip install keras-core

In [ ]:
import tensorflow as tf
import os

In [ ]:
tf.__version__

In [ ]:
DATA_DIR = './'
CLASSES = sorted(os.listdir(DATA_DIR))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# creating the dataset info
data = {
    class_name: len(
        os.listdir(
            os.path.join(DATA_DIR, class_name)
            )
        )
    for class_name in CLASSES
}
ingredients = list(data.keys())
number_of_images = list(data.values())

fig = plt.figure(figsize = (10, 5))

# creating the bar plot
plt.bar(ingredients, number_of_images, color ='maroon',
        width = 0.4)

# Rotation of the bars names
plt.xticks(range(len(ingredients)), ingredients, rotation='vertical')

plt.xlabel("Ingredients")
plt.ylabel("Number of images")
plt.title("Number of images for each ingredient")
plt.show()

In [ ]:
train_ds, val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    labels="inferred",
    label_mode="int",
    class_names=CLASSES,
    color_mode="rgb",
    batch_size=64,
    image_size=(224, 224),
    shuffle=True,
    validation_split=0.1,
    subset="both",
    seed=100
)

In [ ]:
sample_data = list(train_ds.take(1))

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 10))
for images, labels in sample_data:
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(CLASSES[int(labels[i])])
        plt.axis("off")

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import regularizers
import keras_cv

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 10))
for images, labels in sample_data:
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        img = layers.RandomBrightness(factor=0.4, value_range=(0, 255))(images[i])
        plt.imshow(img.numpy().astype("uint8"))
        plt.title(CLASSES[int(labels[i])])
        plt.axis("off")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 10))
for images, labels in sample_data:
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        img = layers.RandomFlip("horizontal")(images[i])
        plt.imshow(img.numpy().astype("uint8"))
        plt.title(CLASSES[int(labels[i])])
        plt.axis("off")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 10))
for images, labels in sample_data:
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        img = layers.RandomRotation(0.5)(images[i])
        plt.imshow(img.numpy().astype("uint8"))
        plt.title(CLASSES[int(labels[i])])
        plt.axis("off")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 10))
for images, labels in sample_data:
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        img = keras_cv.layers.ChannelShuffle()(images[i])
        plt.imshow(img.numpy().astype("uint8"))
        plt.title(CLASSES[int(labels[i])])
        plt.axis("off")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 10))
for images, labels in sample_data:
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        img = keras_cv.layers.GridMask()(images[i])
        plt.imshow(img.numpy().astype("uint8"))
        plt.title(CLASSES[int(labels[i])])
        plt.axis("off")

In [ ]:
data_augmentation = keras.Sequential(
    [
        layers.RandomBrightness(factor=0.4, value_range=(0, 255)),
        layers.RandomFlip("horizontal"),
        layers.RandomFlip("vertical"),
        layers.RandomRotation(0.5),
        keras_cv.layers.ChannelShuffle(),
        keras_cv.layers.GridMask(),
    ]
)

# Apply `data_augmentation` to the training images.
train_ds = train_ds.map(
    lambda img, label: (data_augmentation(img), label),
    num_parallel_calls=tf.data.AUTOTUNE,
)
# Prefetching samples in GPU memory helps maximize GPU utilization.
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.prefetch(tf.data.AUTOTUNE)

In [ ]:
L1_LAMBDA = 1e-5
L2_LAMBDA = 1e-5
IMAGE_SIZE = (224, 224)

def make_model(input_shape, num_classes):
    inputs = keras.Input(shape=input_shape)

    # Entry block
    x = layers.Rescaling(1.0 / 255)(inputs)

    # backbone
    x = layers.Conv2D(32, 3, strides=1, padding="same", kernel_regularizer=regularizers.L2(L2_LAMBDA))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("gelu")(x)
    x = layers.Conv2D(32, 3, strides=1, padding="same", kernel_regularizer=regularizers.L2(L2_LAMBDA))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("gelu")(x)
    x = layers.Conv2D(64, 3, strides=2, padding="same", kernel_regularizer=regularizers.L2(L2_LAMBDA))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("gelu")(x)

    for size in [64, 128, 256, 512]:
        x = layers.Conv2D(size, 3, kernel_regularizer=regularizers.L2(L2_LAMBDA), padding="same")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("gelu")(x)

        x = layers.Conv2D(size, 3, kernel_regularizer=regularizers.L2(L2_LAMBDA), padding="same")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("gelu")(x)

        x = layers.Conv2D(size, 3, kernel_regularizer=regularizers.L2(L2_LAMBDA), padding="same")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("gelu")(x)

        x = layers.MaxPooling2D(3, strides=2)(x)

    # x = layers.BatchNormalization()(x)

    x = layers.GlobalMaxPooling2D()(x)

    # x = layers.Dropout(0.3)(x)
    x = layers.Dense(256, activation="gelu",
                      #kernel_regularizer=regularizers.L1(L1_LAMBDA),
                      activity_regularizer=regularizers.L2(L2_LAMBDA)
                      )(x)

    # x = layers.Dropout(0.3)(x)
    x = layers.Dense(256, activation="gelu",
                      #kernel_regularizer=regularizers.L1(L1_LAMBDA),
                      activity_regularizer=regularizers.L2(L2_LAMBDA)
                      )(x)

    # x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    return keras.Model(inputs, outputs)
model = make_model(input_shape=IMAGE_SIZE + (3,), num_classes=30)

In [ ]:
epochs = 30

callbacks = [
    keras.callbacks.ModelCheckpoint("./"),
]
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
history = model.fit(
    train_ds,
    epochs=epochs,
    callbacks=callbacks,
    validation_data=val_ds
)

In [ ]:
from matplotlib import pyplot as plt
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper left')
plt.show()

In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper left')
plt.show()

# Inference

In [ ]:
import tensorflow as tf
from tensorflow import keras

In [ ]:
img = keras.utils.load_img(
    "./", target_size=(225, 225)
)
# code to use in server starts from here
img_array = keras.utils.img_to_array(img)
img_array = tf.expand_dims(img_array, 0)  # Create batch axis

loaded_model = tf.keras.models.load_model("./")
predictions = loaded_model.predict(img_array)

from pprint import pprint
dist = {CLASSES[i]: predictions[0][i] for i in range(len(CLASSES))}
pprint(dist)

c = CLASSES[np.argmax(predictions[0])]
print(c)

# Test

In [ ]:
loaded_model = tf.keras.models.load_model("./")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(21, 21))
for images, labels in val_ds.take(1):
    for i in range(6*6):
        ax = plt.subplot(6, 6, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        img_array = tf.expand_dims(images[i].numpy().astype("uint8"), 0)  # Create batch axis
        preds = loaded_model.predict(img_array)
        plt.title(CLASSES[np.argmax(preds[0])])
        plt.axis("off")